# Tutorial: Making image cutouts from the data volume (construct the table to do so)

<br>

<img style="position: absolute; right: 60px; top: 40px; height: 160px; width: 160px" src="https://datalabs.esa.int/datalab-icon/c11e47af-7ce9-4281-b3b5-475a180a256f" /> 

**Original Author(s):** Kristin Anett Remmelgas, Héctor Cánovas Cabrera

Notebook from Euclid team altered for SBF specific computations. Find the original in the Euclid example notebooks.

* The notebook uses functions from the **cutout_utils.py Python file**. When moving the notebook you need to move this file as well. You are welcome to modify the utility file for any custom/advanced behaviour you may need.
* The notebook is intended to be run using the **EUCLID-TOOLS kernel** that has all the necessary packages included. 
***

In [ ]:
# to make sure the cutout_utils.py file is updated here after every change
%load_ext autoreload
%autoreload 2

In [ ]:
# importing necessary packages
from astroquery.esa.euclid.core import EuclidClass
import cutout_utils_v2 as ut
import astropy.units as u
from astropy.coordinates import SkyCoord
from astroquery.simbad import Simbad
from astropy.table import Table
import matplotlib.image as mpimg
from astropy.io import fits
from matplotlib import rcParams
from skimage import io
import glob, os, getpass
import pandas as pd

import matplotlib.pyplot as plt
from astropy.visualization import astropy_mpl_style, ImageNormalize, ZScaleInterval, PercentileInterval, AsinhStretch, SqrtStretch, SquaredStretch
plt.style.use(astropy_mpl_style)

In [ ]:
# #making a folder for the future cutouts (if not made already)
# cutout_folder= '/media/user/input_data/test_files/cutouts/'
# if not os.path.exists(cutout_folder):
#     os.makedirs(cutout_folder)

### Choosing the archive environment and login

It is possible to login using many different options - see the # [Euclid.login](https://astroquery.readthedocs.io/en/latest/api/astroquery.esa.euclid.EuclidClass.html#astroquery.esa.euclid.EuclidClass.login) method for details. Similarly to the archive website for safety reasons this login expires after a period of time. Once this happens the queries to the archive will fail (seemingly without reason). If you are running a personal notebook that requires you to stay logged in for longer - make sure to execute this code line again every once in a while. 

In [ ]:
#choosing the archive envionment
env = 'IDR' #only one needed for SBF
t_prefix = 'dr1.'
# setting the environment name for the notebook
Euclid = EuclidClass(environment=env)

# Login ============================
user     = input("Enter your user name: ")   # Alternatively, you may set your user in the line below.
# user     = ''                    
password = getpass.getpass(prompt='Enter your password: ')
Euclid.login(user = user, password = password)


# setting the environment name in the backend too
ut.set_env(env, user = user, password = password)

## 1.1 Getting sources to make cutouts for

The first step is to create a table of sources to start making cutouts for. We will look at 3 possible ways to make this table. The third option is the default (others are commented out) because it does not require any user action but you are welcome to follow the instructions to use the option that works best for you.

* Creating a source table manually (for small amounts of sources)
* Getting a source table from an existing file
* Creating a source table by querying the catalogue

This source table will later be used as part of an ADQL query and in order to do that we need to upload the table as a user_table through Astroquery for the first two options. Since the third option is an ADQL query as well then we can just combine them and the upload step is not necessary.

### Creating a source table manually

In this example we will manually create an Astropy table of sources. The table will only have two columns for each source: "right_ascension" and "declination". For the next steps to work the columns have to be called either that or "ra" and "dec". The table is created from a list of ra and dec coordinates. As a helper function we also demonstrate how to query Simbad for the coordinates of a source by name and then use these coordinates in our table.

In [ ]:
# creating an Astropy table
# if env == 'PDR' or env == 'OTF':
#     simbad_out = Simbad.query_object('NGC6505')
#     ra = [simbad_out['ra'][0], 275.790900]
#     dec = [simbad_out['dec'][0], 67.134113]
# elif env == 'REG':
#     ra = [274.3105620]
#     dec = [64.5661157]
    
# source_table = Table([ra, dec], names=('right_ascension', 'declination'))
# source_table

We now upload the table to your user area as a user table. If you want you can see your user tables on the [archive website](https://eas.esac.esa.int/sas/) under Search -> ADQL form -> User tables. Note that if you already have a table with this name then the line below will give an error and you should change the name.

In [ ]:
# Euclid.upload_table(upload_resource=source_table, table_name='my_sources')

### Getting sources from an existing user file

Alternatively, you may want to use a source table that exists on your computer (or in your workspace in ESA Datalabs). The source table in your file needs to have columns called either "right_ascension" and "declination" or "ra" and "dec" for the source coordinates. It may have any number of other columns as well. This option requires some user interaction from your side. In order to use a file you already have you first need to upload it to ESA Datalabs and then in the cells below provide a path to where you uploaded your file.

The upload function allows formats like: fits, csv, ecsv, VOTable, ASCII and when naming your table please avoid using upper case letters.

In [ ]:
# upload file as user table directly
# user_filepath = "/media/user/example_notebook_outputs/async_result.fits"
# Euclid.upload_table(upload_resource=user_filepath, table_name="my_sources_from_file", format='fits')

In case the cell above does not work or you prefer a different approach you can alternatively upload the file as a user table through the [archive website](https://easreg.esac.esa.int/sas/). In that case make sure your user table is uploaded to the correct archive instance (PDR, OTF, REG) as the user areas of each instance are separate. 

### Getting sources from a catalogue query - default

In the third option we are creating a table of sources by querying the Euclid MER catalogue for sources matching a certain criteria. Please make sure your query has brackets around it (as it will be used as a part of a bigger query in the next step) and contains right_ascension and declination as column names (currently the rest of the code won't work if you change the names). Also if the sources that you would like to make cutouts for are from the MER catalogue be sure to include the column segmentation_map_id in your query - this allows for matching the sources to their corresponding files faster in the next step.

In [ ]:
# note that the names of tables may change between different archive environments
if env == 'PDR':
    mer_cat_name = 'mer_catalogue'
    flux_col = 'flux_vis_1fwhm_aper'
    t_prefix = 'q1.'
if env == 'IDR':
    mer_cat_name = 'mer_catalogue_deep'    
    flux_col = 'flux_vis_1fwhm_aper'
    t_prefix = 'dr1.'
elif env == 'REG':
    mer_cat_name = 'mer_final_catalog_fits_file_regreproc1_r2'
    flux_col = 'flux_vis_aper'
    t_prefix = 'sedm.'
elif env == 'OTF':
    mer_cat_name = 'mer_catalogue'
    flux_col = 'flux_vis_aper'
    t_prefix = 'sedm.'

In [ ]:
#coming up with a suitable query
query = f"""(SELECT right_ascension, declination, segmentation_area, {flux_col}, ellipticity, kron_radius, segmentation_map_id
        FROM {mer_cat_name}
        WHERE segmentation_area < 6100 AND segmentation_area > 5500 AND {flux_col} > 20 AND {flux_col} < 100)"""

job = Euclid.launch_job_async(query)
#adding an order/sorting for easier comparison with other tables later
df_sources = job.get_results().to_pandas().sort_values("right_ascension", ascending= False, ignore_index=True)

print("Found", len(df_sources), " query results")
df_sources[:20]

## 1.2 Getting corresponding files 

The next step is to find corresponding files to make cutouts from for each of the sources in your source table. To do this we have provided a function called `getFiles` in the cutout_util.py helper file that is used in the cell below. As mentioned before the sources can be provided either as the query constructed in the cell above (third option) or using a custom user table you have uploaded as a user table (first and second option). Here is a quick overview of the function parameters:

* product_type (str, mandatory)- type of files you want to make cutouts from. Either mosaic or calibrated_frame
* sources (str, mandatory) - either the query constructed above as str or the name of the uploaded user table (see example)
* instrument_name (str) - either 'VIS' or 'NISP' (specifying NISP will obtain files for all 3 bands unless nisp_filters parameter specified)
* nisp_filters (list, optional) - list containing any or all of the NISP filter names you want NIR_H, NIR_J, NIR_Y
* segmentation_map_id (bool, optional)- whether your source table has this column (enables faster file matching for mosaics, default False)

**What is segmentation_map_id?**

The segmentation_map_id is one of the columns in the MER source catalogue and the first 9 characters of it give the index of the tile where the source was extracted from. This way we can find the correct file directly otherwise we would match with all files that have the source in the field of view and then (in case there are multiple) choose the one where the source is the closest to the image centre.

**Result table**

The result of this getFiles function is a table where the first half of each line is the source information from before and the second half is the file information (obtained from sedm.mosaic_product data table) that will later be used to make a cutout. For VIS there is exactly one line per source but for NISP the same source info will appear 3 times with 3 different file infos (for each of the bands). In the cell below we also sort the table as the last step. This is only necessary so the user can visually compare the table with only sources (from the query) and the resulting table with matching files. For doing the cutouts in the next step the order of the table does not matter so this step can also be removed to save time.

**Note: Paths**

In the result table below the column called 'datalabs_path' shows the directory of the file in the volume mounted in ESA Datalabs. We will be using this path combined with the filename to access the data. The 'file_path' column is the directory of the data in the SAS archives. They are similar but not exactly the same.

In [ ]:
# example 1: query option with VIS files
df_sources_w_files = ut.getFiles(product_type = "mosaic", sources=query, instrument_name='VIS', segmentation_map_id = True, prefix=t_prefix)

# example 2: user table option with 2 NISP bands
# user_table = "user_yourusername.yourtablename" # (e.g. "user_johnsmith.mysources")
# df_sources_w_files = ut.getFiles(product_type = "mosaic", sources=user_table, instrument_name='NISP', nisp_filters = ['NIR_H', 'NIR_Y'], segmentation_map_id = False)

#sorting the table like before for visual comparison
ra_colname = "ra" if "ra" in list(df_sources_w_files.columns) else "right_ascension"
df_sources_w_files = df_sources_w_files.sort_values(ra_colname, ascending= False, ignore_index=True)
df_sources_w_files[:10]

### Checking if there were sources that don't have a matching file

There are some sources that exist in the MER catalogue that might not have an existing mosaic tile at all. These sources automatically do not appear in the table that includes files. In the next cell we double check if this has occurred and for which sources by comparing the initial source table to the one with files.

Note: when using an uploaded custom source table please ignore this part for now

In [ ]:
dif_ra = list(set(df_sources['right_ascension']) - set(df_sources_w_files['right_ascension']))
print("Found", len(dif_ra), "sources without a matching mosaic file")
df_sources[df_sources['right_ascension'].isin(dif_ra)]

## 1.3 Making the cutouts

As the next step we iterate over the table with files and for each line make a cutout using the file path and name. For this we have created a function called `makeCutoutsByTile` in the cutout_utils.py file and here is a quick overview of the function parameters:

* df (mandatory) - table that includes sources and their file info - output of getFiles function
* output_location (mandatory) - path where to store the cutouts
* cutout_size (mandatory) - input size to [astropy.Cutout2D](https://docs.astropy.org/en/stable/api/astropy.nddata.utils.Cutout2D.html). Can be anything astropy Cutout2D allows or in case you want variable cutout size the name of the column from the source table that the cutout size should depend on (see example)
* cutout_func (mandatory if cutout_size is string)- in case of variable cutout size this is the name of a user defined function how the size depends on the selected column
* optional_cutout_params (optional) - other inputs to [astropy.Cutout2D](https://docs.astropy.org/en/stable/api/astropy.nddata.utils.Cutout2D.html) like mode and fill value. The default mode is trim so when a source is on the edge of the image the cutout will be smaller.
* resize_params (optional) - in case you want to resize the cutout before saving it then the inputs to [skimage resize](https://scikit-image.org/docs/stable/api/skimage.transform.html#skimage.transform.resize). By default no resizing is done.


In case you want the cutouts to have variable size depending on one of the source parameters then the source table created in the first step of the notebook needs to have that parameter as a column. The example below will use kron_radius.

Note that the order of making the cutouts will not be the same as the order of sources/files in the table. All the sources that are on the same file will be done together so each file only needs to be read in once.

In [ ]:
# example 1: 0.2 arcmin square cutouts
ut.makeCutoutsByTile(df_sources_w_files[:10], output_location = cutout_folder, cutout_size = 0.2 * u.arcmin)

# example 2: 20 x 30 pixel rectangular cutouts
# ut.makeCutoutsByTile(df_sources_w_files[:10], output_location = cutout_folder, cutout_size = (20, 30))

# example 3: variable cutout size - the cutout size is dependent on the kron_radius of the source using a user-defined function
# in this case cutout size will be source kron_radius * 0.2 / 195 u.arcmin
# def func(x):
#     return x * 0.2 / 195 * u.arcmin
# ut.makeCutoutsByTile(df_sources_w_files[:10], output_location = cutout_folder, cutout_size = 'kron_radius', cutout_func=func)

# example 4: adding more cutout options (default mode is 'trim' with no fill value - the cutouts will differ in size)
# cut_params = {'mode': 'partial', 'fill_value': None}
# ut.makeCutoutsByTile(df_sources_w_files[:10], output_location = cutout_folder, cutout_size = 0.2 * u.arcmin, optional_cutout_params=cut_params)

# example 5: adding parameters for resizing the cutout
# params = {'output_shape': (50, 50), 'order': None, 'mode': 'reflect', 'cval': 0, 'clip': True, 'preserve_range': False, 'anti_aliasing': None, 'anti_aliasing_sigma': None}
# ut.makeCutoutsByTile(df_sources_w_files[:10], output_location = cutout_folder, cutout_size = 'kron_radius', cutout_func=func, resize_params=params)

### Looking at the cutouts

Here we take a quick look at some of the cutouts we have made

In [ ]:
#showing 10 of the cutouts
files = sorted(glob.glob(cutout_folder + "*MOSAIC-VIS*.fits")) #for mosaic VIS files
# files = sorted(glob.glob(cutout_folder + "*MOSAIC-NIR*.fits")) #for mosaic NISP files

_, axes = plt.subplots(2, 5, figsize=(25, 10))
plt.subplots_adjust(wspace=0.1, hspace=0.1)
for ax, img_path in zip(axes.flatten(), files):
    image_data = fits.getdata(img_path, ext=0)
    ax.imshow(image_data, cmap='gray', origin='lower', norm=ImageNormalize(image_data, interval=PercentileInterval(99.5), stretch=AsinhStretch()))
    ax.axis("off")

This part of the notebook shows how to make cutouts for calibrated frames - the steps and used functions are all the same as before but things are happening a bit differently in the cutout_utils.py backend when it comes to calibrated frames. To give a bit of context below is an example of a Euclid calibrated nisp frame. As you can see it is divided into 4x4 detectors with gaps between them. On a file level these detectors are all different extensions of a fits file. This means that when making a cutout we first have to select which detector we are going to cut. 

Another difference to mosaic files is that there is no shortcut (segmentation_map_id) that can be used to match sources to files so we have to do that using the alternate approach of checking whether the source is in the field of fiew of the image.

In [ ]:
rcParams['figure.figsize'] = [5.,5.]
image = io.imread("http://st-dm.pages.euclid-sgs.uk/data-product-doc/dmq1/_images/CCD_numbering1.png")
plt.imshow(image)
plt.axis('off');

### Coming up with a suitable query

Using a similar query as before. If the query fails always make sure you are still logged in.

In [ ]:
#coming up with a suitable query
query = f"""(SELECT right_ascension, declination, segmentation_area, {flux_col}, ellipticity, kron_radius, segmentation_map_id
        FROM {mer_cat_name}
        WHERE segmentation_area < 6100 AND segmentation_area > 5500 AND {flux_col} > 20 AND {flux_col} < 100)"""

job = Euclid.launch_job_async(query)
#adding an order/sorting for easier comparison with other tables later
df_sources = job.get_results().to_pandas().sort_values("right_ascension", ascending= False, ignore_index=True)
print("Found", len(df_sources), " query results")
df_sources[:10]

### Getting corresponding files 

Note that the resulting table below contains all the calibrated frames where the specific source is in the fov of the image. When matching sources with files we make a circle with a default size of 0.5 arcmin centered on the source coordinates and return files where this area is fully in the file fov of the image. Because of the nature of calibrated frames this however does not mean that it is possible to make a cutout of that source from all of those images because the images also contain gaps between the detectors. So when the source is in the gap no cutout is made - that is why the resulting number of cutouts will be less than the number of files in this table.

In [ ]:
# example 1: query option with NISP files
df_sources_w_files_calib = ut.getFiles(product_type = "calibrated_frame", sources=query, instrument_name='NISP', nisp_filters = ['NIR_H'], prefix=t_prefix)
# example 2: user table option with VIS files
# user_table = "user_yourusername.yourtablename" # (e.g. "user_johnsmith.mysources")
# df_sources_w_files_calib = ut.getFiles(product_type = "calibrated_frame", sources=user_table, instrument_name='VIS')

#sorting the table like before for visual comparison
ra_colname = "ra" if "ra" in list(df_sources_w_files_calib.columns) else "right_ascension"
df_sources_w_files_calib = df_sources_w_files_calib.sort_values(ra_colname, ascending= False, ignore_index=True)
df_sources_w_files_calib[:10]

### Making the cutouts

Other than choosing the correct detector for making the cutout this part works the same as before.

In [ ]:
# example 1: 0.2 arcmin square cutouts
ut.makeCutoutsByTile(df_sources_w_files_calib[:15], output_location = cutout_folder, cutout_size = 0.2 * u.arcmin)

# example 2: 20x30 pixel rectangular cutouts
# ut.makeCutoutsByTile(df_sources_w_files_calib[:10], output_location = cutout_folder, cutout_size = (20, 30))

# example 3: the cutout radius becomes dependent on the kron_radius of the source
# def func(x):
#     return x * 0.2 / 195 * u.arcmin
# ut.makeCutoutsByTile(df_sources_w_files_calib[:10], output_location = cutout_folder, cutout_size = 'kron_radius', cutout_func=func)

# example 4: adding more cutout options (default mode is 'trim' with no fill value - the cutouts will differ in size)
# cut_params = {'mode': 'partial', 'fill_value': None}
# ut.makeCutoutsByTile(df_sources_w_files_calib[:10], output_location = cutout_folder, cutout_size =  0.2 * u.arcmin, optional_cutout_params=cut_params)

# example 5: adding parameters for resizing the cutout
# params = {'output_shape': (50, 50), 'order': None, 'mode': 'reflect', 'cval': 0, 'clip': True, 'preserve_range': False, 'anti_aliasing': None, 'anti_aliasing_sigma': None}
# ut.makeCutoutsByTile(df_sources_w_files_calib[:10], output_location = cutout_folder, cutout_size = 'kron_radius', cutout_func=func, resize_params=params)

### Looking at the cutouts

Here we take a quick look at some of the cutouts we have made

In [ ]:
#showing 10 of the cutouts
files = sorted(glob.glob(cutout_folder + "*NIR_W-CAL*.fits")) #for calibrated NISP files
# files = sorted(glob.glob(cutout_folder + "*VIS_SWL*.fits")) #for calibrated VIS files

_, axes = plt.subplots(2, 5, figsize=(25, 10))
plt.subplots_adjust(wspace=0.1, hspace=0.1)
for ax, img_path in zip(axes.flatten(), files):
    image_data = fits.getdata(img_path, ext=0)
    ax.imshow(image_data, cmap='gray', origin='lower', norm=ImageNormalize(image_data, interval=PercentileInterval(99.5), stretch=AsinhStretch()))
    ax.axis("off")

# Stacked frames

This next section of the notebook focuses on making cutouts from stacked frame products. The overall logic is very similar to the mosaic case. However please note that since **stacked frame products are not present in all archive environments** (not in PDR for example) this part can only work in the environments where they are available.

### Uploading a table -- from Vizier catalog

In [ ]:
from astroquery.vizier import Vizier

In [ ]:
Vizier.ROW_LIMIT = -1

catalog = Vizier.get_catalogs("catalogname")

In [ ]:
print(catalog.keys())
table = catalog[0]
print(table.colnames)
table

# ---------------- 
# sidetrack to just gest vizier table

df = catalog[0].to_pandas()
df
#--------------------

In [ ]:
# Optional RA,Dec convertion to decimal degrees 

def ra_to_decimal(ra):
    h, m, s = map(float, str(ra).split())
    return 15 * (h + m/60 + s/3600)

def dec_to_decimal(dec):
    d, m, s = map(float, str(dec).split())
    
    sign = -1 if d < 0 else 1
    d = abs(d)
    
    return sign * (d + m/60 + s/3600)

# To use run below

# df['right_ascension'] = df['RAJ2000'].apply(ra_to_decimal)
# df['declination'] = df['DEJ2000'].apply(dec_to_decimal)

# df

In [ ]:
table_name = 'your_table_name'  #Do not use capital letters

# make sure to use the right columns. Target is not that necessary, can be index instead. right_ascension and declination obviously are, but re; effective radius is also important, used to find the appropriate cutout size
source_table_stack = Table([df['target'], df['right_ascension'], df['declination'],df['re']],
                          names=('target', 'right_ascension', 'declination', 're'))
source_table_stack

### Uploading a table manually -- from csv file

In [ ]:
# uploading table manual table from csv
table_name = 'your_table_name'

df = pd.read_csv('path_to_your_csv.csv')
df.keys()

In [ ]:
# make sure to use the right columns. Target is not that necessary, can be index instead. right_ascension and declination obviously are, but re; effective radius is also important, used to find the appropriate cutout size
source_table_stack = Table([df['target'], df['right_ascension'], df['declination'],df['re']],
                          names=('target', 'right_ascension', 'declination', 're'))
source_table_stack

### Uploading the created table

Uploading a user table will raise a warning if you already have a table with the same name uploaded. In that case you can use a different name or delete the existing one (see the commented examples in the cell below).

In [ ]:
# Get user tables =====================
tables  = Euclid.load_tables(only_names=True)
utables = [inp.get_qualified_name() for inp in tables if 'user_' in inp.get_qualified_name()]
utables = [inp[inp.find('.')+1:] for inp in utables]

In [ ]:
utables

In [ ]:
# upload user table (if table upload fails try again after deleting existing table with same name)
# table_name defined above
if table_name in utables:
    print(f'Warning: {table_name} exist in user space - creating: {table_name}_1')
    table_name = f'{table_name}_1'

Euclid.upload_table(upload_resource=source_table_stack, table_name = table_name)

In [ ]:
# Incase something went wrong:
# To delete a table uncomment this line and fill in <table_name> with the appropriate table name
# Euclid.delete_user_table(table_name=table_name) 

### Getting corresponding files 

Note that the cell below automatically gets your Datalabs user name (same as cosmos user name) from an environment variable.

In [ ]:
table_name = 'your_table_name'

In [ ]:
# example 1: user table option with VIS files 
user_table               = f"user_{os.environ.get('USER')}.{table_name}"  # (e.g. "user_johnsmith.my_sources_stack_cutout_nb")
user_table

In [ ]:
# Choose which filterband (if both just do one at a time)
filt = 'VIS'
# filt = 'H'

In [ ]:
# now actually doing the matching
if filt == 'VIS':
    df_sources_w_files_stack, df_res_dup = ut.getFiles(product_type = "stacked_frame", sources=user_table, instrument_name='VIS', segmentation_map_id = False, prefix=t_prefix, return_duplicate_table=True)

elif filt == 'H':
    # example 2: user table option with 2 NISP bands || original comment, check out how to do multiple nisp bands in one go
    df_sources_w_files_stack, df_res_dup = ut.getFiles(product_type = "stacked_frame", sources=user_table, instrument_name='NISP', nisp_filters = ['NIR_H'], segmentation_map_id = False, prefix=t_prefix, return_duplicate_table=True)

#sorting the table like before for visual comparison
ra_colname = "ra" if "ra" in list(df_sources_w_files_stack.columns) else "right_ascension"
df_sources_w_files_stack = df_sources_w_files_stack.sort_values(ra_colname, ascending= False, ignore_index=True)
df_res_dup = df_res_dup.sort_values(ra_colname, ascending= False, ignore_index=True)
df_sources_w_files_stack[:10]

### Redoing the distance selection

In [ ]:
import numpy as np
def point_to_segment_distance(point, a, b):
    """Distance from a point to a line segment."""

    point = np.asarray(point, dtype=float)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    ab = b - a
    ap = point - a

    denom = np.dot(ab, ab)
    if denom == 0:
        return np.linalg.norm(ap)

    t = np.dot(ap, ab) / denom
    t = np.clip(t, 0.0, 1.0)

    closest = a + t * ab
    return np.linalg.norm(point - closest)


def border_margin(row, radec_colnames):
    """
    Distance from the source to the nearest image edge.
    Larger values mean the source is further from the border.
    """

    point = np.array([
        row[radec_colnames["ra_colname"]],
        row[radec_colnames["dec_colname"]],
    ])

    corners = np.asarray(row["fov"], dtype=float).reshape(4, 2)

    edges = [
        (corners[0], corners[1]),
        (corners[1], corners[2]),
        (corners[2], corners[3]),
        (corners[3], corners[0]),
    ]


    return min(
        point_to_segment_distance(point, a, b)
        for a, b in edges
    )


def chooseBestFile(df_res_dup, radec_colnames,
                   check_border_cases=True,
                   dist_center_border=0.4,plot=False, do_print=True, tag_old_oid=False):
    """
    Choose the best file for each source/filter combination.

    First uses the radial distance to the image centre.
    If the source is far from the centre, compare the best two
    candidates using the distance to the nearest image edge.
    """
    grouped = df_res_dup.groupby(
        [
            radec_colnames["ra_colname"],
            radec_colnames["dec_colname"],
            "filter_name",
        ],
        sort=False,
    )

    chosen_rows = []
    # if plot:
        

    for _, group in grouped:

        group = group.sort_values("dist").reset_index(drop=True)

        best = group.iloc[0].copy()

        # Only one matching frame
        if len(group) == 1:
            chosen_rows.append(best)
            continue

        if check_border_cases and (best["dist"] > dist_center_border):
            if do_print:
                print(f"Performing check for {best['target']}")

            second = group.iloc[1].copy()

            margin_best = border_margin(best, radec_colnames)
            margin_second = border_margin(second, radec_colnames)

            if margin_second > margin_best:
                if do_print:
                    print(
                        f"Border case: for {best['target']} replacing "
                        f"tile {best['observation_id']} with {second['observation_id']}"
                    )
                if plot:
                    fig,ax = plt.subplots(figsize=(4,4))
                    ax.plot(best['fov'][0::2], best['fov'][1::2], color='blue')
                    ax.plot(second['fov'][0::2], second['fov'][1::2], color='red')
                    ax.scatter(best[radec_colnames["ra_colname"]], best[radec_colnames["dec_colname"]], color='black')
                    plt.show()
                    
                if tag_old_oid:
                    old_oid = best['observation_id']
                    
                best = second
                
                if tag_old_oid:
                    best['reselected_oid_previously'] = old_oid

        chosen_rows.append(best)
    # if plot:
        
    return pd.DataFrame(chosen_rows).reset_index(drop=True)

In [ ]:
# this part is defined in the environment, so we need it again here
ra_colname, dec_colname = 'right_ascension', 'declination'
radec_colnames = {"ra_colname": ra_colname, "dec_colname": dec_colname}

# so that we can test the new function
df_final = chooseBestFile(df_res_dup, radec_colnames, plot=False, do_print=True, tag_old_oid=True)

In [ ]:
df_final

### Checking if there are any sources without matching stacked frame files

In [ ]:
table = source_table_stack.to_pandas()
dif_ra = list(set(table['right_ascension']) - set(df_sources_w_files_stack['right_ascension']))
print("Found", len(dif_ra), "sources without a matching stacked file")
table[table['right_ascension'].isin(dif_ra)]

In [ ]:
# inspect galaxies that are not matched --> outside of Euclid sky area
fig = plt.figure()
plt.scatter(table['right_ascension'], table['declination'],marker='.', color='blue')
plt.scatter(table[table['right_ascension'].isin(dif_ra)]['right_ascension'], table[table['right_ascension'].isin(dif_ra)]['declination'], marker='.', color='red')
plt.show()
plt.close()

### Writing the found files to a table

In [ ]:
# If distance selection was not redone use
# df_sources_w_files_stack.to_csv(f'/media/user/youroutputplace/{table_name}_{filt}_matched.csv', index=False)


# but incase of the reselection based on distance we use instead
df_final.to_csv(f'/media/user/youroutputplace/{table_name}_{filt}_matched.csv', index=False)

In [ ]:
Euclid.logout()

### Making the cutouts ! Done in other notebook !

***

**Runnning the notebook outside of ESA Datalabs**

The current notebook was developed with the intent to use it inside ESA Datalabs, so all the filepaths point to the data volume mounted in ESA Datalabs. 

----------------------------------------------------------------